In [52]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, OrdinalEncoder
from sklearn.model_selection import cross_val_score,cross_validate

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [53]:
# import data and take a look at few rows
original_df = pd.read_csv('train.csv')
original_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [54]:
# check shape, null values and weight in case dtypes need to be changed .info()
original_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


The dataframe is light, there is no need to change Dtype of variables to make them lighter.

Some variables have null values. We can check how many of them exactly.

In [55]:
# Zoom in on null values
original_df.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [56]:
original_df.describe().round(2)

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.00,891.00,891.00,714.00,891.00,891.00,891.00
mean,446.00,0.38,2.31,29.70,0.52,0.38,32.20
std,257.35,0.49,0.84,14.53,1.10,0.81,49.69
min,1.00,0.00,1.00,0.42,0.00,0.00,0.00
25%,223.50,0.00,2.00,20.12,0.00,0.00,7.91
50%,446.00,0.00,3.00,28.00,0.00,0.00,14.45
75%,668.50,1.00,3.00,38.00,1.00,0.00,31.00
max,891.00,1.00,3.00,80.00,8.00,6.00,512.33


In [57]:
# Make sure every row is a different observation and check how many different values each variable has
original_df.nunique()

PassengerId    891
Survived         2
Pclass           3
Name           891
Sex              2
Age             88
SibSp            7
Parch            7
Ticket         681
Fare           248
Cabin          147
Embarked         3
dtype: int64

Letter and number of cabin might relate to the location of the cabin, characteristics of the dweller or any other circumstance which could be related to the chances of survaving (maybe disable people where assigned a certain type of cabin and they were given priority to disembark regardless of their gender). I wonder if extracting the letter and number of cabin or only letter could be useful.

In [58]:
# split data into X and y
X = original_df.drop(columns=['PassengerId', 'Survived', 'Name', 'Ticket'])
y = original_df['Survived']

In [59]:
# keep a map of index vs PassengerId
passenger_id = original_df['PassengerId']

##  Set baseline-first development

### Transformations needed at this stage

Columns with missing values to be imputed:  
> Age (apply median and create a flag)  
> Cabin ('M' for missing)  
> Embarked
---
Columns to apply a function:  
> Cabin (extract code of cabin and recategorize G and T to M)
---
Categorical columns to be encoded:
> Pclass  
> Cabin  
> Embarked  

Inspect Cabin feature codes

In [60]:
# fill missing values, extract code and check value counts
X['Cabin'].fillna('M').str.extract(r'([A-Z])').value_counts()

0
M    687
C     59
B     47
D     33
E     32
A     15
F     13
G      4
T      1
Name: count, dtype: int64

Cabin feature:

There were 687 missing values, therefore, M is none of the extracted letters.

Values G and T don't have enough observations to keep a category (less than 1%), move them to the missing category

In [61]:
# check value counts of Embarked
X['Embarked'].value_counts()

Embarked
S    644
C    168
Q     77
Name: count, dtype: int64

Embarked feature:

There are only 2 missing values. It's not enough to create another category. They should be assigned to the categorical mode.

#### Transform columns

In [62]:
X.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked
0,3,male,22.0,1,0,7.2500,NaN,S
1,1,female,38.0,1,0,71.2833,C85,C
2,3,female,26.0,0,0,7.9250,NaN,S
3,1,female,35.0,1,0,53.1000,C123,S
4,3,male,35.0,0,0,8.0500,NaN,S


In [63]:
# transformer pipelines
cabin_transformer = Pipeline(steps=[
    ('imputer_c', SimpleImputer(strategy='constant', fill_value='M')),
    ('regex', FunctionTransformer(lambda x: x.iloc[:, 0].str.extract(r'([A-Z])', expand=True), feature_names_out='one-to-one')), # function transformers are not official so don't have a built in specified output name, this makes the transformer to output the input name if get_feature_names_out is used in the column transformer
    ('recategorize', FunctionTransformer(lambda x: x.replace(['G', 'T'], 'M'), feature_names_out='one-to-one')), # same as above
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

age_transformer = Pipeline(steps=[
    ('imputer_a', SimpleImputer(strategy='mean', add_indicator=True))
])

embarked_tranformer = Pipeline(steps=[
    ('imputer_e', SimpleImputer(strategy='constant', fill_value='S')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

sex_transformer = Pipeline(steps=[
    ('encoder', OrdinalEncoder())
])

pclass_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# column transformer
preprocessor = ColumnTransformer(
    transformers=(
        ('cabin', cabin_transformer, ['Cabin']),
        ('age', age_transformer, ['Age']),
        ('embarked', embarked_tranformer, ['Embarked']),
        ('sex', sex_transformer, ['Sex']),
        ('pclass', pclass_transformer, ['Pclass'])
    ),
    remainder='passthrough'
)

preprocessor.set_output(transform='pandas');

Check that intended transformations take place

In [64]:
# Uncomment the code below

# transformations_check = preprocessor.fit_transform(X)
# print(transformations_check.nunique())
# transformations_check

### Fit tree based models on the data

In [74]:
# create a dictionary of tree based models to try
tree_models = {
    'RandomForest':RandomForestClassifier(random_state=42),
    'ExtraTree':ExtraTreesClassifier(random_state=42),
    'HistGradientBoosting':HistGradientBoostingClassifier(random_state=42),
    'XGB':XGBClassifier(),
    'CatBoost':CatBoostClassifier(random_state=42, verbose=0)
}

#### Using cross_val_score()

In [78]:
# create a variable to add the model scores (accuracy)
model_scores = {}

# loop through the tree_models dictionary 
for name, model in tree_models.items():
    # model pipeline
    clf = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    cross_val_mean = np.mean(cross_val_score(clf, X, y, cv=5, n_jobs=-1))
    model_scores[name] = cross_val_mean

In [81]:
pd.Series(model_scores).to_frame('first_eda')

,first_eda
RandomForest,0.803634
ExtraTree,0.778940
HistGradientBoosting,0.838409
XGB,0.820470
CatBoost,0.824926
